# JavaScript Network Requests

A comprehensive guide to making HTTP requests in JavaScript, covering the Fetch API, Axios, and XMLHttpRequest — with real-world patterns for GET/POST requests, error handling, request cancellation, streaming, and WebSockets. All examples use a browser-based app but the patterns apply equally to Node.js.

## Table of Contents

- [Module 01: HTTP Requests with fetch()](#module-01-http-requests-with-fetch)
- [Module 02: Deep Dive into fetch()](#module-02-deep-dive-into-fetch)
- [Module 03: HTTP Requests with Axios](#module-03-http-requests-with-axios)
- [Module 04: HTTP Requests with XMLHttpRequest](#module-04-http-requests-with-xmlhttprequest)
- [Module 05: WebSockets](#module-05-websockets)

Diagrams are rendered inline below each relevant section.

## Module 01: HTTP Requests with fetch()

_Introduces the Fetch API with a brief HTTP fundamentals refresher, and demonstrates the simplest possible `fetch()` call for retrieving JSON data._

### Course Overview

This course covers the following approaches to making network requests in JavaScript:

| Module | Topic |
|--------|-------|
| 01 | HTTP fundamentals + Fetch API |
| 02 | Deep dive into fetch() |
| 03 | Axios HTTP client |
| 04 | XMLHttpRequest |
| 05 | WebSockets |

> _This course uses JavaScript in the browser. Everything covered is equally applicable to Node.js environments._

**Version notes:**
- Written using ECMAScript 2022, Node 18.x LTS
- Fully applicable to ECMAScript 2015–2022 and Node 17.5–19.x

### HTTP Fundamentals

HTTP is a request/response protocol. The client sends an HTTP request; the server returns an HTTP response.

![HTTP request/response diagram](assets/01-http-request-response.png)

**Request structure** — the client sends:
- An HTTP method + path + protocol version: `GET /api/events?page=2 HTTP/2`
- Headers like `Host`, `User-Agent`, `Cookie`

**Response structure** — the server returns:
- A status line: `HTTP/2 200 OK`
- Headers like `Content-Type`, `Content-Length`
- A response body (e.g. JSON)

### Simple HTTP Requests with fetch()

The Fetch API provides a modern, Promise-based interface for HTTP requests. The minimal pattern:

```js
fetch('api/events')
    .then(response => response.json())
    .then(data => console.log(data));
```

- `fetch(url)` returns a Promise that resolves to a `Response` object.
- `.json()` reads the response body and parses it as JSON — it also returns a Promise.
- Chaining `.then()` handles the parsed data asynchronously.

### Demo: Globoticket Application

A simple Express + SQLite app that serves a ticket-listing page. The front-end uses a single `fetch('api/events')` call to load event data from the server and render it into a table.

See [`01-fetch-demos/`](01-fetch-demos/) for the full project (Express server + browser front-end).

## Module 02: Deep Dive into fetch()

_Explores the full `fetch()` options object, POST requests, error handling, request abortion with `AbortController`, streaming responses, and CORS._

### fetch() Options

`fetch()` accepts an optional second argument — a configuration object:

```js
fetch(
    'api/events',
    {
        method: 'GET',
        headers: {
            Accept: 'text/xml'
        }
    }
);
```

### POST Requests

To send data to the server, set `method: 'POST'`, provide a `Content-Type` header, and pass a JSON-serialised body:

```js
fetch(
    'api/cart',
    {
        method: 'POST',
        headers: {
            'Content-Type': 'application/json'
        },
        body: JSON.stringify(
            { id: 42, quantity: 1 }
        )
    }
);
```

### Error Scenarios

Three distinct failure modes exist when using `fetch()`:

| Scenario | Behaviour |
|---|---|
| Error thrown by `fetch()` | Network failure — Promise rejects |
| HTTP status code indicating an error (4xx / 5xx) | Promise **resolves** — `response.ok` is `false` |
| Error while parsing response (e.g. invalid JSON) | `.json()` Promise rejects |

The most common mistake is assuming a 4xx/5xx response rejects the `fetch()` promise — it does not. Always check `response.ok` or `response.status`:

```js
fetch('api/events')
    .then(response => {
        if (response.status !== 200) {
            showError('Something went wrong.');
        }
        return response.json();
    })
    .then(events => showEvents(events))
    .catch(ex => console.error(ex));
```

### HTTP Status Code Ranges

| Range | Meaning |
|---|---|
| 1xx | Informational |
| 2xx | Success |
| 3xx | Redirection |
| 4xx | Client errors |
| 5xx | Server errors |

### Aborting Requests with AbortController

`AbortController` allows in-flight requests to be cancelled programmatically. The controller produces a `signal` that is passed to `fetch()`. Calling `controller.abort()` cancels the request.

![AbortController usage pattern](assets/02-abort-controller.png)

```js
// Create one controller per request to allow independent cancellation
let controllers = [];
events.forEach(event => {
    const controller = new AbortController();
    controllers.push(controller);
    fetch(`api/cover/${event.id}`, { signal: controller.signal })
        .then(response => response.text())
        .then(img => showImage(img, event.id))
        .catch(ex => console.log(ex)); // AbortError lands here
});

// Cancel all pending requests
function abort() {
    controllers.forEach(controller => controller.abort());
}
```

### Streaming Responses with ReadableStream

The `response.body` property exposes a `ReadableStream`. Instead of waiting for the entire response, data can be consumed as it arrives in **chunks** — useful for large payloads or progress tracking.

A `ReadableStream` is a sequence of data chunks delivered over time. A reader can consume the stream chunk by chunk via `reader.read()`, which returns a Promise resolving to `{ done, value }` where `value` is a `Uint8Array`.

```js
fetch('api/cover/42')
    .then(response => {
        const reader = response.body.getReader();
        const chunks = [];
        function read() {
            reader.read().then(({ done, value }) => {
                if (done) {
                    showImage(decodeChunks(chunks), 42);
                    return;
                }
                chunks.push(value);
                showProgress(chunks.length, 42);
                read(); // read next chunk recursively
            });
        }
        read();
    });
```

### Cross-Origin Resource Sharing (CORS)

Browsers enforce the **same-origin policy**: a page at `http://app.example.com` cannot freely request resources from `http://api.example.com`. CORS is the mechanism that allows servers to explicitly permit cross-origin access.

![CORS cross-origin request](assets/03-cors-cross-origin.png)

The browser automatically adds an `Origin` header to cross-origin requests. If the server responds with `Access-Control-Allow-Origin` matching the requesting origin, the browser permits the JavaScript code to read the response.

### CORS Preflight Request

For non-simple requests (e.g. POST with `Content-Type: application/json`), the browser first sends an `OPTIONS` **preflight** request to check permissions before sending the actual request.

![CORS preflight request](assets/04-cors-preflight.png)

The preflight response must include:
- `Access-Control-Allow-Origin` — permitted origins
- `Access-Control-Allow-Methods` — permitted HTTP methods
- `Access-Control-Allow-Headers` — permitted request headers

Only after a successful preflight does the browser send the actual POST request.

### Demo: Deep Dive into fetch()

A multi-stage demo covering HTTP headers, POST requests, error handling, aborting in-flight requests, and streaming image progress. Each subfolder (`1_HTTP header`, `2_POST request`, `4_error handling`, `5_aborting requests`, `6_progress`) is a self-contained Express app.

See [`02-fetch-deep-dive-demos/`](02-fetch-deep-dive-demos/) for the full project.

## Module 03: HTTP Requests with Axios

_Covers Axios — a popular third-party HTTP client that works in both browser and Node.js — with GET, POST, error handling, and download progress._

### Installing Axios

Three ways to include Axios:

| Method | Code |
|---|---|
| Package manager + ESM import | `import axios from 'axios'` |
| Package manager + CommonJS | `const axios = require('axios')` |
| Script tag | `<script src="axios.js">` |

### GET Requests with Axios

```js
// Axios wraps the response — data lives at response.data
axios.get('api/events')
    .then(response => showEvents(response.data));
```

Key difference from `fetch()`: Axios automatically parses JSON and places the result in `response.data`. No separate `.json()` call is needed.

### POST Requests with Axios

```js
function addToCart(id, quantity) {
    axios({
        method: 'POST',
        url: 'api/cart',
        headers: { 'Content-Type': 'application/json' },
        data: { id: id, quantity: quantity }
    })
        .then(response => {
            if (response.status === 200) {
                axios.get('api/cart')
                    .then(response => updateCart(response.data));
            }
        });
}
```

### Error Handling with Axios

Unlike `fetch()`, Axios **rejects** the Promise for HTTP error status codes (outside 200–299). This makes error handling more consistent:

| Error condition | Axios behaviour |
|---|---|
| HTTP status outside 200–299 | Promise rejects |
| No response from server | Promise rejects |
| Malformed JSON | Promise rejects |

```js
axios.get('api/events')
    .then(response => showEvents(response.data))
    .catch(error => console.error(error));
```

### Download Progress with Axios

Axios exposes `onDownloadProgress` for tracking download progress. The callback receives a progress event with a `progress` property (0–1) when the `Content-Length` header is known, or `undefined` otherwise:

```js
axios({
    url: 'api/cover/42',
    method: 'GET',
    onDownloadProgress: (e) => {
        console.log(e.progress ? 100 * e.progress : 'N/A', '%');
    }
});
```

### Demo: HTTP Requests with Axios

A multi-stage demo covering GET requests, POST requests, error handling, and download progress tracking using Axios with the Globoticket Express app. Each subfolder (`2_get`, `3_post`, `4_error`, `5_progress`) is a self-contained Express app.

See [`03-axios-demos/`](03-axios-demos/) for the full project.

## Module 04: HTTP Requests with XMLHttpRequest

_Covers the original browser HTTP API — `XMLHttpRequest` (XHR) — including its history, core methods and properties, GET/POST requests, and request timeouts._

### History of XMLHttpRequest

XHR is the original mechanism for asynchronous browser HTTP requests, predating the Fetch API by over a decade.

![XMLHttpRequest history timeline](assets/05-xhr-history-timeline.png)

| Year | Milestone |
|------|----------|
| 2001 | Internet Explorer 5.0 ships XHR as an ActiveX object |
| 2002 | Other browsers ship native XHR support |
| 2005 | Jesse J. Garrett coins the term **Ajax** (Asynchronous JavaScript and XML) |

### Key XMLHttpRequest Methods

| Method | Purpose |
|---|---|
| `xhr.open('GET', 'api/events')` | Configure the HTTP request (method + URL) |
| `xhr.send()` | Send the HTTP request |
| `xhr.abort()` | Abort the HTTP request |

### Key XMLHttpRequest Properties

| Property | Description |
|---|---|
| `readyState` | Request lifecycle state (0–4) |
| `status` | HTTP response status code |
| `statusText` | HTTP response status text |
| `responseText` | Response body as a string |
| `responseXML` | Response body parsed as XML |

### GET Requests with XMLHttpRequest

```js
const xhr = new XMLHttpRequest();
xhr.open('GET', 'api/events', true); // true = asynchronous
xhr.onload = () => {
    const events = JSON.parse(xhr.responseText);
    showEvents(events);
};
xhr.send(null);
```

### POST Requests with XMLHttpRequest

```js
function addToCart(id, quantity) {
    const xhrPOST = new XMLHttpRequest();
    xhrPOST.open('POST', 'api/cart', true);
    xhrPOST.setRequestHeader('Content-Type', 'application/json');
    xhrPOST.onload = () => {
        const xhrGET = new XMLHttpRequest();
        xhrGET.open('GET', 'api/cart', true);
        xhrGET.onload = () => {
            const cart = JSON.parse(xhrGET.responseText);
            updateCart(cart);
        };
        xhrGET.send(null);
    };
    xhrPOST.send(JSON.stringify({ id: id, quantity: quantity }));
}
```

### Request Timeouts

XHR has built-in timeout support via the `timeout` property (in milliseconds). If the request takes longer than `timeout`, the `ontimeout` event fires instead of `onload`.

![XHR timeout — onload vs ontimeout](assets/06-xhr-timeout.png)

```js
const xhr = new XMLHttpRequest();
xhr.open('GET', 'api/events-slow', true);
xhr.timeout = 1000; // milliseconds
xhr.onload = () => {
    const events = JSON.parse(xhr.responseText);
    showEvents(events);
};
xhr.ontimeout = () => {
    showError('Request timed out!');
};
xhr.send(null);
```

### Demo: HTTP Requests with XMLHttpRequest

A multi-stage demo covering GET requests, POST requests, and timeout handling with XHR against the Globoticket Express app. Each subfolder (`3_get`, `4_post`, `5_timeout`) is a self-contained Express app.

See [`04-xhr-demos/`](04-xhr-demos/) for the full project.

## Module 05: WebSockets

_Introduces WebSockets — a full-duplex, persistent connection protocol that enables real-time push communication between client and server without repeated HTTP polling._

### Why WebSockets?

HTTP is inherently request/response — the client must initiate every exchange. WebSockets establish a **persistent, bidirectional channel** over a single TCP connection. Once the connection is open, either side can send messages at any time.

**Typical use cases:** live notifications, chat, real-time dashboards, collaborative editing, live cart updates.

### Browser WebSocket API

The browser exposes a `WebSocket` constructor. The connection lifecycle is event-driven:

```js
const ws = new WebSocket('ws://localhost:3000');

ws.onopen = () => {
    console.log('Connection established');
};

ws.onmessage = (event) => {
    const cart = JSON.parse(event.data);
    updateCart(cart);
};

ws.onerror = (error) => {
    console.error('WebSocket error:', error);
};

ws.onclose = () => {
    console.log('Connection closed');
};

ws.send(JSON.stringify({ action: 'subscribe' }));
```

### WebSocket Server with Node.js `ws`

The `ws` package is the standard Node.js WebSocket library. It can be combined with an Express HTTP server so both HTTP and WebSocket connections share the same port:

```js
const express = require('express');
const ws = require('ws');
const app = express();

// Share one HTTP server between Express and ws
const httpServer = require('http').createServer();
const wss = new ws.Server({ server: httpServer });

// Associate each WebSocket connection with the user's session
wss.on('connection', (conn, req) => {
    const cookies = cookie.parse(req.headers.cookie);
    conn.sid = cookieParser.signedCookie(cookies['connect.sid'], secret);

    conn.on('message', (message) => {
        conn.send(`Echo: ${message}`);
    });
});

// Push a cart update to all WebSocket clients with matching session
wss.clients.forEach((client) => {
    if (client.sid === req.session.id &&
        client.readyState === ws.WebSocket.OPEN) {
        client.send(JSON.stringify([{ quantity: row['count'] }]));
    }
});

httpServer.on('request', app);
httpServer.listen(3000);
```

### Cart Update Pattern

In the demo application, when a user adds an item to their cart via a POST request, the server immediately **pushes** the updated cart count to all WebSocket connections belonging to that session — so other open tabs update in real time without polling.

### Demo: WebSockets

A full Globoticket Express app with a WebSocket server (`ws` package) sharing port 3000 with Express. The server pushes cart count updates via WebSocket whenever a POST to `/api/cart` succeeds, enabling real-time sync across browser tabs for the same session.

See [`05-websockets-demos/`](05-websockets-demos/) for the full project.